# legal-rag · Build dữ liệu (Kaggle)

Một notebook cho cả luồng build, **dùng chính code của repo** (giống hệt code đang chạy production):

```
Bộ Pháp điển (HF, ghim revision) ─► chunk theo cấu trúc ─► embedding AITeamVN (2×T4)      ─► index/
Hỏi đáp Bộ Tư pháp (HF)          ─► tạo đáp án từ trích dẫn + kiểm chứng nội dung          ─► golden_vK.jsonl.gz
index + golden ─► chấm 494 câu test đúng cấu hình production ─► CỔNG: nDCG@10 ≥ 0,40 và không tụt > 0,01 so với champion
   đạt + PUBLISH ─► Qdrant phapdien_vN + MinIO (bảng từ BM25, bộ câu hỏi) + MLflow (run + Model Registry: candidate)
   ─► bạn sửa data.version: vN trong values.yaml, push ─► Jenkins thấy cấu hình đã ĐẠT trong MLflow ─► deploy ─► champion
```

**Chạy:** Settings → Accelerator **GPU T4 x2**, **Internet on**. Add-ons → **Secrets** (chỉ cần khi `PUBLISH = True`):

| Secret | Giá trị |
|---|---|
| `QDRANT_URL`, `QDRANT_API_KEY` | Qdrant, key có quyền **ghi** |
| `S3_ENDPOINT`, `S3_ACCESS_KEY`, `S3_SECRET_KEY` | MinIO của legal-rag (`https://s3.oci.lexsy.pro.vn`), user có quyền ghi bucket `legal-rag` |
| `MLFLOW_TRACKING_URI`, `MLFLOW_TRACKING_USERNAME`, `MLFLOW_TRACKING_PASSWORD` | `https://mlflow.oci.lexsy.pro.vn` + tài khoản |
| `GITHUB_TOKEN` | chỉ khi repo private |

Thời gian: ~30–40 phút embedding + ~5 phút tạo đáp án + ~3 phút chấm. Bị ngắt thì chạy lại, phần embedding chạy tiếp từ shard đã xong.

In [ ]:
# 1. Lấy code từ repo (clone vào /tmp để token không lọt vào Output của notebook)
import os, sys, subprocess
REPO, BRANCH = "https://github.com/liesyx/VN-Legal-Retriever-Eval", "main"

def secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return os.environ.get(name)

for k in ["QDRANT_URL", "QDRANT_API_KEY", "S3_ENDPOINT", "S3_ACCESS_KEY", "S3_SECRET_KEY",
          "MLFLOW_TRACKING_URI", "MLFLOW_TRACKING_USERNAME", "MLFLOW_TRACKING_PASSWORD"]:
    v = secret(k)
    if v:
        os.environ[k] = v
print("secrets:", sorted(k for k in os.environ if k.startswith(("QDRANT_", "S3_", "MLFLOW_"))))

SRC = "/tmp/legal_rag"
tok = secret("GITHUB_TOKEN")
url = REPO.replace("https://", f"https://x-access-token:{tok}@") if tok else REPO
if not os.path.exists(SRC):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCH, url, SRC], check=True)
COMMIT = subprocess.run(["git", "-C", SRC, "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
sys.path.insert(0, SRC)
# Kaggle đã có torch, sentence-transformers, pandas, scikit-learn; chỉ thêm client Qdrant + S3
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "qdrant-client==1.19.1", "boto3"], check=True)
print("code:", REPO, BRANCH, COMMIT)

In [ ]:
# 2. Cấu hình — chỉ sửa ở đây
from pathlib import Path

DATA_VERSION   = 2        # -> collection phapdien_v2. Luôn tăng, không ghi đè bản đang chạy (rollback = đổi lại số cũ)
GOLDEN_VERSION = 1        # bộ câu hỏi chuẩn eval/golden_v1.jsonl.gz
BUILD_GOLDEN   = True     # True: dựng bộ câu hỏi từ dataset hỏi đáp (lần đầu, hoặc muốn bộ mới -> tăng GOLDEN_VERSION)
                          # False: tải bộ có sẵn trên MinIO (các lần build dữ liệu sau: chấm trên cùng bộ để so được với champion)
PUBLISH        = False    # True: đạt cổng -> đẩy Qdrant + MinIO + ghi MLflow để Jenkins deploy được
LIMIT_ROWS     = None     # 2000 = chạy thử nhanh (không bao giờ publish)
MIN_NDCG, MAX_DROP = 0.40, 0.01          # = quality.minNdcg10 / maxDrop trong values.yaml

EMB_MODEL_REF = "models/vietnamese-embedding/main"   # = model.embedding trong values.yaml (model production đọc từ MinIO)
S3_BUCKET     = "legal-rag"
QA_ID, QA_CFG = "tmquan/pbgdpl-vn-legal-qna", "qa"

OUT, INDEX = Path("/kaggle/working/out"), Path("/kaggle/working/out/index")
OUT.mkdir(parents=True, exist_ok=True)
COLLECTION = f"phapdien_v{DATA_VERSION}"
GOLDEN_REF = f"eval/golden_v{GOLDEN_VERSION}.jsonl.gz"
GOLDEN = OUT / f"golden_v{GOLDEN_VERSION}.jsonl.gz"
os.environ.update(EMB_MODEL_REF=EMB_MODEL_REF, GOLDEN_REF=GOLDEN_REF, EVAL_SAMPLE="0", APP_VERSION=f"kaggle-{COMMIT}")
if LIMIT_ROWS:
    PUBLISH = False

import torch
print(COLLECTION, "·", GOLDEN_REF, "· publish:", PUBLISH, "· GPU:", torch.cuda.device_count())
assert torch.cuda.device_count(), "Chưa bật GPU: Settings → Accelerator → GPU T4 x2"

## 3. Index: chunk + embedding

In [ ]:
import json
from datasets import load_dataset
from huggingface_hub import HfApi
from tqdm.auto import tqdm
from pipeline.embed import IndexConfig, build_index
from pipeline.index_data import load_index

icfg = IndexConfig()
api = HfApi()
DATASET_REV, MODEL_REV = api.dataset_info(icfg.dataset_id).sha, api.model_info(icfg.model_id).sha
print("dataset", DATASET_REV, "· model", MODEL_REV)

man = json.loads((INDEX / "manifest.json").read_text()) if (INDEX / "manifest.json").exists() else {}
if man.get("dataset", {}).get("revision") == DATASET_REV and man.get("model", {}).get("revision") == MODEL_REV and not LIMIT_ROWS:
    chunks, E = load_index(INDEX)                       # đã build ở lần chạy trước (cùng dữ liệu, cùng model)
    print("dùng lại index:", len(chunks), "chunk")
else:
    df = load_dataset(icfg.dataset_id, icfg.dataset_cfg, split="train", revision=DATASET_REV).to_pandas()
    if LIMIT_ROWS:
        df = df.sample(LIMIT_ROWS, random_state=42)
    chunks, E = build_index(df, INDEX, icfg, dataset_rev=DATASET_REV, model_rev=MODEL_REV, progress=tqdm)
    del df
for f in sorted(INDEX.iterdir()):
    print(f"  {f.name:22s} {f.stat().st_size/1e6:8.1f} MB")

## 4. Bộ câu hỏi chuẩn (golden set)

In [ ]:
import boto3
from botocore.config import Config as BotoConfig
from mlops import golden, labels

def s3():
    return boto3.client("s3", endpoint_url=os.environ["S3_ENDPOINT"], aws_access_key_id=os.environ["S3_ACCESS_KEY"],
                        aws_secret_access_key=os.environ["S3_SECRET_KEY"], region_name="us-east-1",
                        config=BotoConfig(signature_version="s3v4", s3={"addressing_style": "path"}))

def s3_exists(key):
    try:
        s3().head_object(Bucket=S3_BUCKET, Key=key); return True
    except Exception:
        return False

GOLDEN_STATS = {}
if BUILD_GOLDEN:
    qa_rev = api.dataset_info(QA_ID).sha
    qa = load_dataset(QA_ID, QA_CFG, split="train", revision=qa_rev).to_pandas()
    Q, GOLDEN_STATS = labels.build_golden(chunks, qa)          # seed 42, dev 30% — như benchmark
    GOLDEN_STATS["qa_revision"] = qa_rev
    Q.to_parquet(OUT / "qrels.parquet", index=False)
    golden.export(str(OUT / "qrels.parquet"), str(GOLDEN))
    (OUT / "golden_stats.json").write_text(json.dumps(GOLDEN_STATS, ensure_ascii=False, indent=2, default=str))
    print(json.dumps({k: v for k, v in GOLDEN_STATS.items() if k != "citation_reasons"}, ensure_ascii=False, default=str))
else:
    s3().download_file(S3_BUCKET, GOLDEN_REF, str(GOLDEN))
    print("tải", GOLDEN_REF)
items = golden.load(str(GOLDEN))
print(f"chấm trên {len(items)} câu test")

## 5. Chấm đúng cấu hình production + cổng chất lượng

In [ ]:
import pandas as pd
from legal_core.search import Config
from mlops import evaluate as ev
from pipeline.file_retriever import FileRetriever

cfg = Config(device="cuda", depth=0)            # = production (RERANK=0): hybrid 0,7·dense + 0,3·BM25
fr = FileRetriever(cfg, str(INDEX))
rows, S = ev.evaluate(fr, items)
DESC = ev.describe(cfg, COLLECTION)             # cùng "vân tay" với Job cổng chất lượng trên k3s
OK, WHY, CHAMP = ev.gate(S, DESC, MIN_NDCG, MAX_DROP)

# Báo cáo có biểu đồ + giải thích (cùng trang này được lưu vào MLflow: run -> Artifacts -> report.html)
from IPython.display import HTML
from mlops import report
display(HTML(report.build(S, rows, DESC, "passed" if OK else "failed", WHY, min_ndcg=MIN_NDCG,
                          champion=CHAMP and CHAMP["metrics"].get(ev.MAIN), title=f"Chấm {COLLECTION} trên Kaggle")))

## 6. Đẩy lên production (chỉ khi đạt và `PUBLISH = True`)

In [ ]:
from legal_core.search import qdrant_client
from pipeline.publish_qdrant import publish

STATUS, published = ("passed" if OK else "failed"), False
if PUBLISH and OK:
    up_golden = BUILD_GOLDEN
    if BUILD_GOLDEN and s3_exists(GOLDEN_REF):            # chạy lại ô này: cùng nội dung thì thôi, khác thì không ghi đè
        s3().download_file(S3_BUCKET, GOLDEN_REF, str(OUT / "golden_remote.jsonl.gz"))
        same = golden.load(str(OUT / "golden_remote.jsonl.gz"), None) == golden.load(str(GOLDEN), None)
        if not same:
            raise SystemExit(f"{GOLDEN_REF} đã có trên MinIO với nội dung khác — không ghi đè. Tăng GOLDEN_VERSION hoặc BUILD_GOLDEN=False")
        up_golden = False
    client = qdrant_client(os.environ["QDRANT_URL"], os.environ["QDRANT_API_KEY"], timeout=120)
    vocab = OUT / f"bm25_vocab_v{DATA_VERSION}.json.gz"
    publish(client, str(INDEX), DATA_VERSION, str(vocab), Config())          # chạy lại được: tự đẩy tiếp nếu bị ngắt
    s3().upload_file(str(vocab), S3_BUCKET, f"data/{COLLECTION}/bm25_vocab.json.gz")
    if up_golden:
        s3().upload_file(str(GOLDEN), S3_BUCKET, GOLDEN_REF)
    published = True
    print(f"✓ Qdrant {COLLECTION} · s3://{S3_BUCKET}/data/{COLLECTION}/bm25_vocab.json.gz" + (f" · s3://{S3_BUCKET}/{GOLDEN_REF}" if BUILD_GOLDEN else ""))
elif not OK:
    print("Không đạt cổng -> không đẩy gì. Production giữ nguyên.")
else:
    print("PUBLISH = False -> chỉ ghi thí nghiệm vào MLflow.")

# MLflow: chỉ ghi gate=passed (Jenkins sẽ bỏ qua bước chấm lại) khi dữ liệu ĐÃ nằm trên Qdrant
if OK and not published:
    STATUS = "experiment"
RUN_ID = ev.record(DESC, S, rows, STATUS, WHY, name=f"build {COLLECTION} (Kaggle)", out=str(OUT / "eval"),
                   tags={"purpose": "build", "evaluated_on": "kaggle-file", "dataset_rev": DATASET_REV, "model_rev": MODEL_REV,
                         "code_commit": COMMIT, "golden_built": str(BUILD_GOLDEN),
                         **({"golden_with_gold": GOLDEN_STATS.get("with_gold"), "golden_test": GOLDEN_STATS.get("test")} if GOLDEN_STATS else {})})
print("MLflow run:", RUN_ID or "(không ghi được — xem cảnh báo)")

## 7. Bước tiếp theo

In [ ]:
if published:
    print(f'''Xong. Lên production:
  1. helm/legal-rag/values.yaml:   data.version: v{DATA_VERSION}
                                    quality.enabled: true
                                    quality.golden: {GOLDEN_REF}
  2. commit + push main -> Jenkins -> Job cổng chất lượng thấy cấu hình đã ĐẠT trong MLflow (run {RUN_ID}) -> deploy
  3. deploy xong: alias champion trong MLflow = phiên bản này. Quay lại bản cũ: đổi data.version về số cũ (collection cũ vẫn còn).''')
else:
    print("Chưa đẩy gì lên production. Output (index, golden, eval) nằm trong tab Output của notebook.")